In [2]:
import torch

## Tensors of different ranks

In [ ]:
def print_tensor_properties(x, name=""):
    print(f"\n##### {name}")
    print(f"Tensor: {x}")
    print(f"Shape: {tuple(x.shape)}")
    print(f"Dim: {x.dim()}")
    print(f"Numel: {x.numel()}") # total number of elements
    print(f"dtype: {x.dtype}")
    print(f"device: {x.device}")
    print(f"requires_grad: {x.requires_grad}")

print_tensor_properties(torch.rand(()), "SCALAR (0D)")
print_tensor_properties(torch.rand(2), "VECTOR (1D)")
print_tensor_properties(torch.rand(2,2), "MATRIX (2D)")
print_tensor_properties(torch.rand(2,2,2), "3D TENSOR")
print_tensor_properties(torch.rand(2,2,2,2), "4D TENSOR")


##### SCALAR (0D)
Tensor: 0.48747164011001587
Shape: ()
Dim: 0
Numel: 1
dtype: torch.float32
device: cpu
requires_grad: False

##### VECTOR (1D)
Tensor: tensor([0.1371, 0.0118])
Shape: (2,)
Dim: 1
Numel: 2
dtype: torch.float32
device: cpu
requires_grad: False

##### MATRIX (2D)
Tensor: tensor([[0.2530, 0.2487],
        [0.5429, 0.2653]])
Shape: (2, 2)
Dim: 2
Numel: 4
dtype: torch.float32
device: cpu
requires_grad: False

##### 3D TENSOR
Tensor: tensor([[[0.8878, 0.7362],
         [0.5800, 0.0567]],

        [[0.8694, 0.3724],
         [0.6297, 0.8568]]])
Shape: (2, 2, 2)
Dim: 3
Numel: 8
dtype: torch.float32
device: cpu
requires_grad: False

##### 4D TENSOR
Tensor: tensor([[[[0.7819, 0.8529],
          [0.0550, 0.3344]],

         [[0.4293, 0.2752],
          [0.8861, 0.5349]]],


        [[[0.3373, 0.8029],
          [0.0145, 0.3688]],

         [[0.3967, 0.2534],
          [0.8918, 0.2121]]]])
Shape: (2, 2, 2, 2)
Dim: 4
Numel: 16
dtype: torch.float32
device: cpu
requires_grad: False


## Memory layout intuition: stride, storage_offset, transpose effects

1. **(memory) stride():** </br>
how many elements in the underlying storage you jump to move by 1 step along each dimension; For a contiguous (3,4) tensor, PyTorch stores it row-major (C-order), so you get: stride = (4, 1). Meaning:
    - Move by 1 in the last dim (columns): jump 1 element in memory.
    - Move by 1 in the first dim (rows): jump 4 elements in memory (skip one full row)

2. **storage_offset():** </br>
tells you where (at what index) this tensor’s first element starts inside its underlying 1D storage

3. **contiguous tensor:** </br>
a tensor is contiguous if its elements are stored in memory exactly in the order implied by its shape (standard row-major layout, like NumPy).
In other words: no gaps, no reordering — just sequential memory.
Some operations require contiguous memory:
    - A.view() -- get error is not a contiguous tensor
    - to fix this error use A.contiguous() that creates a new tensor with reordered memory so it follows standard layout

In [32]:
A = torch.arange(12).reshape(3, 4)
print("A:\n", A)
print("A.shape:", A.shape)
print("A.stride():", A.stride())
print("A.storage_offset():", A.storage_offset())
print("A contiguous?", A.is_contiguous())

B = A.t()
print("\nB = A.t():\n", B)
print("B.shape:", B.shape)
print("B.stride():", B.stride())
print("B.storage_offset():", B.storage_offset())
print("B contiguous?", B.is_contiguous())

# Show they share the same underlying storage (same data pointer)
# (storage() is OK for learning; untyped_storage() is the newer API)
print("Same storage pointer?", A.storage().data_ptr() == B.storage().data_ptr())

A:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
A.shape: torch.Size([3, 4])
A.stride(): (4, 1)
A.storage_offset(): 0
A contiguous? True

B = A.t():
 tensor([[ 0,  4,  8],
        [ 1,  5,  9],
        [ 2,  6, 10],
        [ 3,  7, 11]])
B.shape: torch.Size([4, 3])
B.stride(): (1, 4)
B.storage_offset(): 0
B contiguous? False
Same storage pointer? True


## View on contiguous tensor

1. **view():** </br>
is used to reshape a tensor without copying the data (same memory, different shape, it is fast because data in not copied)
    - only works when tensor it contiguous

In [34]:
A = torch.arange(12).reshape(3, 4)
print("A:\n", A)
print("A.stride():", A.stride(), "| contiguous?", A.is_contiguous())

V = A.view(2, 6)
print("\nV = A.view(2,6):\n", V)
print("V.stride():", V.stride(), "| contiguous?", V.is_contiguous())
print("A and V share storage?", A.storage().data_ptr() == V.storage().data_ptr())

# Prove it is a view by modifying V and observing A change
V[0, 0] = -999
print("\nAfter V[0,0] = -999:")
print("A:\n", A)
print("V:\n", V)

A:
 tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
A.stride(): (4, 1) | contiguous? True

V = A.view(2,6):
 tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
V.stride(): (6, 1) | contiguous? True
A and V share storage? True

After V[0,0] = -999:
A:
 tensor([[-999,    1,    2,    3],
        [   4,    5,    6,    7],
        [   8,    9,   10,   11]])
V:
 tensor([[-999,    1,    2,    3,    4,    5],
        [   6,    7,    8,    9,   10,   11]])


In [3]:
A = torch.arange(12).reshape(3,4)
B = A.t()

B.view(-1)   # RuntimeError

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

## View on non-contiguous tensor

1. Fix with .contiguous()

2. Fix using reshape() instead of view(): </br>
it will use a view if possible, and copy tha data if necessary (slower if copy happens, more flexible)

In [35]:
A = torch.arange(12).reshape(3, 4)
B = A.t()  # non-contiguous
print("A contiguous?", A.is_contiguous())
print("B = A.t() contiguous?", B.is_contiguous())
print("B.stride():", B.stride())

# view should fail
try:
    C = B.view(12)
    print("C (view) succeeded unexpectedly:", C)
except RuntimeError as e:
    print("view failed as expected:")
    print("  ", e)

# Fix 1: make contiguous then view
B_contig = B.contiguous()
C1 = B_contig.view(12)
print("\nFix 1: B.contiguous().view(12)")
print("B_contig contiguous?", B_contig.is_contiguous())
print("C1:", C1, "| contiguous?", C1.is_contiguous())

# Fix 2: reshape (will return view if possible, else copy)
C2 = B.reshape(12)
print("\nFix 2: B.reshape(12)")
print("C2:", C2, "| contiguous?", C2.is_contiguous())
print("B and C2 share storage?", B.storage().data_ptr() == C2.storage().data_ptr())



A contiguous? True
B = A.t() contiguous? False
B.stride(): (1, 4)
view failed as expected:
   view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

Fix 1: B.contiguous().view(12)
B_contig contiguous? True
C1: tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11]) | contiguous? True

Fix 2: B.reshape(12)
C2: tensor([ 0,  4,  8,  1,  5,  9,  2,  6, 10,  3,  7, 11]) | contiguous? True
B and C2 share storage? False


## Permute dimensions + prove index mapping + strides/contiguity

1. permute()" </br>
reorders dimensions of a tensor. It usually returns a view (no copy), by changing the tensor’s stride (so it’s often non-contiguous afterward).
    - What happens in memory? permute() does not move data (most of the time).
    It changes how indices map to the same storage by changing strides.

In [4]:
X = torch.randn(2, 3, 4) # (batch, channels, width) for example; indx (0,1,2)
print("X.shape:", X.shape)
print("X.stride():", X.stride(), "| contiguous?", X.is_contiguous())

X1 = X.permute(2, 0, 1)  # (2,3,4) -> (4,2,3)
print("\nX1 = X.permute(2,0,1)")
print("X1.shape:", X1.shape)
print("X1.stride():", X1.stride(), "| contiguous?", X1.is_contiguous())

X2 = X.permute(1, 2, 0)  # (2,3,4) -> (3,4,2)
print("\nX2 = X.permute(1,2,0)")
print("X2.shape:", X2.shape)
print("X2.stride():", X2.stride(), "| contiguous?", X2.is_contiguous())

# Prove values moved as expected by checking index mapping
i, j, k = 1, 2, 3
print("\nIndex mapping check:")
print("X[i,j,k] =", X[i, j, k].item())
print("X1[k,i,j] =", X1[k, i, j].item(), " (should match X[i,j,k])")
print("X2[j,k,i] =", X2[j, k, i].item(), " (should match X[i,j,k])")

X.shape: torch.Size([2, 3, 4])
X.stride(): (12, 4, 1) | contiguous? True

X1 = X.permute(2,0,1)
X1.shape: torch.Size([4, 2, 3])
X1.stride(): (1, 12, 4) | contiguous? False

X2 = X.permute(1,2,0)
X2.shape: torch.Size([3, 4, 2])
X2.stride(): (4, 1, 12) | contiguous? False

Index mapping check:
X[i,j,k] = -0.02665996551513672
X1[k,i,j] = -0.02665996551513672  (should match X[i,j,k])
X2[j,k,i] = -0.02665996551513672  (should match X[i,j,k])


## Simulate CNN tensor layout

1. Typical ML use-cases: convert images between NHWC and NCHW: </br>
    - N - batch size, C - channels, H - height, W - width
    - Example (32, 3, 244, 224): 32 images, 3 channels (RGB), 224x224 pixels
    - NHWC = "channel-last"
    - NCHW = "channel-first"
    - NCHW layout PyTorch, NHWC layout TensorFlow/Keras
    - GPU faster with NCHW, TPUs faster with NHWC

Comment to code below: </br>
NCHW → NHWC uses `permute(0,2,3,1)`. Back uses `permute(0,3,1,2)`. Values preserved; contiguity may change because permute changes strides.


In [5]:
images = torch.randn(8, 3, 32, 32)  # NCHW: (N, C, H, W)
print("images (NCHW) shape:", images.shape)

images_nhwc = images.permute(0, 2, 3, 1)  # (N, H, W, C)
print("images_nhwc (NHWC) shape:", images_nhwc.shape)

images_back = images_nhwc.permute(0, 3, 1, 2)  # back to (N, C, H, W)
print("images_back (NCHW) shape:", images_back.shape)

# Sanity check: values preserved
print("Max |diff| after round-trip:", (images - images_back).abs().max().item())
print("NHWC contiguous?", images_nhwc.is_contiguous(), "| back contiguous?", images_back.is_contiguous())


images (NCHW) shape: torch.Size([8, 3, 32, 32])
images_nhwc (NHWC) shape: torch.Size([8, 32, 32, 3])
images_back (NCHW) shape: torch.Size([8, 3, 32, 32])
Max |diff| after round-trip: 0.0
NHWC contiguous? False | back contiguous? True


## Manual broadcasting reasoning

1. Broadcasting </br>
in PyTorch (and NumPy) is a rule that allows tensors with different shapes to participate in element-wise operations by automatically expanding smaller dimensions.
Importantly: no real data is copied — PyTorch just pretends the smaller tensor repeats along some dimensions.

Example:
```
A = torch.tensor([[1,2,3],
                  [4,5,6]])
b = torch.tensor([10,20,30])
C = A + b

Shapes: 
A : (2,3)
b : (3,)

PyTorch treats b as if it were:
[[10,20,30],
 [10,20,30]]

Result:
[[11,22,33],
 [14,25,36]]
 
But no actual copy happens.
```

2. Rules </br>
    - PyTorch compares dimensions from right to left. Two dimensions are compatible if:
        - They are equal, or
        - One of themis 1, or
        - One dimension does not exist.

3. Why needed? It allows vectorized operations like:
    - Add bias in neural networks: X : (batch, features), b : (features,) → X+b
    - Normalize data: X : (batch, features), mean : (features,), std : (features,) → (X-mean)/std


Comment to code below: </br>
`(3,4) + (4,)` works because `(4,)` behaves like `(1,4)` and expands along the missing leading dim → `(3,4)`.

In [8]:
A = torch.randn(3, 4)
b = torch.randn(4)
C = A + b

print("A.shape:", A.shape)
print("b.shape:", b.shape)
print("C.shape:", C.shape)

# Show what broadcasting is doing explicitly:
b_explicit = b.unsqueeze(0).expand_as(A)  # (1,4) -> (3,4) via expansion
# i.e. .unsqeeze(0) adds a new dimension at the front, i.e. turned a 1D vector into a "row"
print("b.unsqueeze(0).shape:", b.unsqueeze(0).shape)
print("b_explicit.shape:", b_explicit.shape)
# expand_as(A) means “expand this tensor to have the same shape as A”.
# So (1,4) becomes (3,4) by repeating along the first dimension virtually (no copy).
print("Max |diff| between (A+b) and (A+b_explicit):", (C - (A + b_explicit)).abs().max().item())
# Why does it work? Because b's shape (4,) is treated as (1,4) 
# and broadcast across rows.
# What dimension expanded? The missing leading dimension expanded from 1 -> 3 (rows).


A.shape: torch.Size([3, 4])
b.shape: torch.Size([4])
C.shape: torch.Size([3, 4])
b.unsqueeze(0).shape: torch.Size([1, 4])
b_explicit.shape: torch.Size([3, 4])
Max |diff| between (A+b) and (A+b_explicit): 0.0


## Broadcasting failure case

Comment to code below: </br>
`(3,4) + (3,)` fails because `(3,)` behaves like `(1,3)` and the last dim mismatches `4 vs 3`. Fix by `b.unsqueeze(1)` → `(3,1)` so it can expand across columns.

In [ ]:
A = torch.randn(3, 4)
b = torch.randn(3)

print("A.shape:", A.shape)
print("b.shape:", b.shape)

try:
    C = A + b
    print("Unexpectedly worked. C.shape:", C.shape)
except RuntimeError as e:
    print("Broadcasting failed as expected:")
    print(" ", e)

# Why fail? Aligning from the right: A is (3,4), b is (3,) -> treated as (1,3).
# Compare last dims: 4 vs 3 mismatch (and neither is 1) => error.
# Fix? Make b (3,1) so last dim is 1 and can expand to 4.

# Fix: if b is meant to add per-row, reshape to (3,1) so it broadcasts across columns
b_fixed = b.unsqueeze(1)  # (3,) -> (3,1)
C_fixed = A + b_fixed

print("b_fixed.shape (after unsqueeze):", b_fixed.shape)
print("C_fixed.shape:", C_fixed.shape)

# Prove it matches explicit expansion
b_rowwise = b_fixed.expand_as(A)  # (3,1) -> (3,4)
print("Max |diff| between (A+b_fixed) and (A+b_rowwise):", (C_fixed - (A + b_rowwise)).abs().max().item())


A.shape: torch.Size([3, 4])
b.shape: torch.Size([3])
Broadcasting failed as expected:
  The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 1
b_fixed.shape (after unsqueeze): torch.Size([3, 1])
C_fixed.shape: torch.Size([3, 4])
Max |diff| between (A+b_fixed) and (A+b_rowwise): 0.0


## Complex broadcasting

Comment to code below: </br>
Treat `B(3,1)` as `(1,3,1)` then expand to `(2,3,4)` by expanding dims `1→2` and `1→4`.

In [10]:
A = torch.randn(2, 3, 4)
B = torch.randn(3, 1)

print("A.shape:", A.shape)
print("B.shape:", B.shape)

# Broadcasting aligns from the right:
# A: (2, 3, 4)
# B:    (3, 1)   -> pretend it is (1, 3, 1) to match rank
# Then expand to (2, 3, 4):
#   dim -3: 1 -> 2
#   dim -2: 3 -> 3 (matches)
#   dim -1: 1 -> 4

B_compat = B.unsqueeze(0)          # (3,1) -> (1,3,1)
B_expanded = B_compat.expand(2, 3, 4)  # (1,3,1) -> (2,3,4)

C = A + B_expanded
print("B_compat.shape:", B_compat.shape)
print("B_expanded.shape:", B_expanded.shape)
print("C.shape:", C.shape)

# Also show PyTorch can do it automatically if shapes are compatible:
C_auto = A + B  # should work because B is treated as (1,3,1)
print("Max |diff| manual vs auto:", (C - C_auto).abs().max().item())

# (Optional) Make the alignment super explicit with a small deterministic example:
A_small = torch.arange(2*3*4).reshape(2,3,4).float()
B_small = torch.tensor([[10.0],[20.0],[30.0]])  # (3,1)
C_small = A_small + B_small  # broadcast
print("\nSmall check:")
print("A_small[0,:,0] =", A_small[0,:,0])
print("B_small[:,0]   =", B_small[:,0])
print("C_small[0,:,0] =", C_small[0,:,0], "  (should be A_small[0,:,0] + B_small[:,0])")

A.shape: torch.Size([2, 3, 4])
B.shape: torch.Size([3, 1])
B_compat.shape: torch.Size([1, 3, 1])
B_expanded.shape: torch.Size([2, 3, 4])
C.shape: torch.Size([2, 3, 4])
Max |diff| manual vs auto: 0.0

Small check:
A_small[0,:,0] = tensor([0., 4., 8.])
B_small[:,0]   = tensor([10., 20., 30.])
C_small[0,:,0] = tensor([10., 24., 38.])   (should be A_small[0,:,0] + B_small[:,0])


## Implement matmul manually (2D)

Comment to code below: </br>
Manual matmul is just the definition: $C_{ij} = \sum_k A_{ik}B_{kj}$. Batched is the same, with an extra batch loop.

In [13]:
def manual_matmul(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """
    Manual matrix multiplication for 2D tensors:
      A: (m, k)
      B: (k, n)
    returns:
      C: (m, n)
    """
    if A.dim() != 2 or B.dim() != 2:
        raise ValueError("manual_matmul expects 2D tensors.")
    m, kA = A.shape
    kB, n = B.shape
    if kA != kB:
        raise ValueError(f"Incompatible shapes: {A.shape} x {B.shape}")

    # allocate output
    C = torch.zeros((m, n), dtype=A.dtype, device=A.device)

    # triple loop
    for i in range(m):
        for j in range(n):
            s = 0.0
            for k in range(kA):
                s += A[i, k].item() * B[k, j].item()
            C[i, j] = s
    return C

A = torch.randn(3, 5)
B = torch.randn(5, 4)

C_manual = manual_matmul(A, B)
C_torch = A @ B  # for verification only 
print("C_manual.shape:", C_manual.shape)
print("Max |diff| vs torch (@):", (C_manual - C_torch).abs().max().item())


C_manual.shape: torch.Size([3, 4])
Max |diff| vs torch (@): 1.1920928955078125e-07


## Manutal batched matmul

In [14]:
# A: (batch, m, k)
# B: (batch, k, n)
# returns: (batch, m, n)

def manual_batched_matmul(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    if A.dim() != 3 or B.dim() != 3:
        raise ValueError("manual_batched_matmul expects 3D tensors (batch, m, k) and (batch, k, n).")
    bA, m, kA = A.shape
    bB, kB, n = B.shape
    if bA != bB or kA != kB:
        raise ValueError(f"Incompatible shapes: {A.shape} x {B.shape}")

    C = torch.zeros((bA, m, n), dtype=A.dtype, device=A.device)

    for b in range(bA):
        for i in range(m):
            for j in range(n):
                s = 0.0
                for k in range(kA):
                    s += A[b, i, k].item() * B[b, k, j].item()
                C[b, i, j] = s
    return C

Ab = torch.randn(2, 3, 5)
Bb = torch.randn(2, 5, 4)

Cb_manual = manual_batched_matmul(Ab, Bb)
Cb_torch = Ab @ Bb  # verification only
print("Cb_manual.shape:", Cb_manual.shape)
print("Max |diff| vs torch batched (@):", (Cb_manual - Cb_torch).abs().max().item())


Cb_manual.shape: torch.Size([2, 3, 4])
Max |diff| vs torch batched (@): 2.384185791015625e-07


## Naive softmax (1D)

Comment to code below: </br>
Naive softmax overflows because `exp(1000)` becomes `inf`. Stable softmax subtracts `max(x)` so the biggest exponent is `exp(0)=1`, preserving the result.

In [15]:
def softmax_naive(x: torch.Tensor) -> torch.Tensor:
    """
    Naive softmax for 1D tensor.
    """
    if x.dim() != 1:
        raise ValueError("softmax_naive expects a 1D tensor.")
    ex = torch.exp(x)
    return ex / ex.sum()

x = torch.tensor([1.0, 2.0, 3.0])
s_naive = softmax_naive(x)
s_ref = torch.softmax(x, dim=0)
print("naive:", s_naive)
print("torch.softmax:", s_ref)
print("Max |diff|:", (s_naive - s_ref).abs().max().item())

naive: tensor([0.0900, 0.2447, 0.6652])
torch.softmax: tensor([0.0900, 0.2447, 0.6652])
Max |diff|: 0.0


## Numerically stable softmax

In [18]:
def softmax_stable(x: torch.Tensor) -> torch.Tensor:
    """
    Numerically stable softmax for 1D tensor.
    """
    if x.dim() != 1:
        raise ValueError("softmax_stable expects a 1D tensor.")
    x_shift = x - x.max()
    ex = torch.exp(x_shift)
    return ex / ex.sum()

x_big = torch.tensor([1000.0, 1001.0, 1002.0])
try:
    s_naive_big = softmax_naive(x_big)
    print("naive on big:", s_naive_big)
except RuntimeError as e:
    print("naive softmax failed on big values:", e)

s_stable_big = softmax_stable(x_big)
s_ref_big = torch.softmax(x_big, dim=0)
print("stable:", s_stable_big)
print("torch.softmax:", s_ref_big)
print("Max |diff|:", (s_stable_big - s_ref_big).abs().max().item())

# Naive softmax fails because exp(1000) overflows to inf in float32/float64,
# causing inf/inf -> NaN or unstable results.
# Subtracting max(x) shifts values so the largest exponent is exp(0)=1,
# avoiding overflow while preserving probabilities.

naive on big: tensor([nan, nan, nan])
stable: tensor([0.0900, 0.2447, 0.6652])
torch.softmax: tensor([0.0900, 0.2447, 0.6652])
Max |diff|: 0.0


## Basic gradient

Comment to code below: </br>
Autograd built a computation graph from x through `**2`, `*2`, and `+`. Backward applies chain rule $dy/dx = 2x + 2.

In [19]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x
y.backward()
print("x:", x.item())
print("y:", y.item())
print("x.grad:", x.grad.item())

# Graph built: x feeds into two branches:
#   (1) x -> square -> x**2
#   (2) x -> multiply by 2 -> 2*x
# then add -> y
# backward() applies chain rule to compute dy/dx = 2x + 2, evaluated at x=3 -> 8.


x: 3.0
y: 15.0
x.grad: 8.0


## Gradient through softmax

Comment to code below: </br>
$\sum_i \text{softmax}(z)_i = 1$ (constant), so gradient is zero (up to numerical noise).

In [20]:
z = torch.randn(5, requires_grad=True)
p = torch.softmax(z, dim=0)
loss = p.sum()          # sum of probabilities
loss.backward()

print("z:", z.detach())
print("softmax(z):", p.detach())
print("loss (sum):", loss.item())
print("z.grad:", z.grad)

# Key observation:
# sum(softmax(z)) == 1 for any z (up to floating point error),
# so its gradient should be ~0.
print("grad norm:", z.grad.norm().item())

# softmax outputs a probability distribution that sums to 1.
# Therefore f(z) = sum_i softmax(z)_i is (theoretically) constant = 1,
# so df/dz = 0.
# In practice you get tiny non-zero grads due to floating point + implementation details.

z: tensor([ 0.3790, -0.0478,  0.7463,  1.5716,  0.4428])
softmax(z): tensor([0.1341, 0.0875, 0.1936, 0.4419, 0.1429])
loss (sum): 1.0000001192092896
z.grad: tensor([-1.5984e-08, -1.0431e-08, -2.3078e-08, -5.2678e-08, -1.7037e-08])
grad norm: 6.294569487863555e-08


## Summary

In this notebook you can find the answers for the following questions:
1. Why does view fail on transpose?
2. What does contiguous actually mean?
3. Why does broadcasting align from the right?
4. Why subtract max in softmax?
5. What is stored during autograd forward pass?